In [ ]:
from pathlib import Path
import os
from functools import partial
import multiprocessing as mp
from typing import Callable

import sympy
import numpy as np
from scipy.optimize import minimize
from scipy.stats import wasserstein_distance
import matplotlib.pyplot as plt

from make_and_load_models import get_timeseries_and_KM
from utils import kl_divergence, jeffreys_divergence, SteadyFP

In [ ]:
SCRATCH_PATH = Path(f"/home/zachuu/scratch/paper/")
SCRATCH_PATH.mkdir(parents=True, exist_ok=True)
FIG_PATH = SCRATCH_PATH

In [ ]:
equation_names = [
    "Double Well",
    "Triple Well",
    "Softglass (Pitchforking)",
    "Softglass (Normal)",
]
folders = [
    "DoubleWell",
    "TripleWell",
    "SoftglassPitchforking",
    "SoftglassNormal",
]
drift_coefficients = [
    # 1, x, x|x|, x^3, x^3|x|, ...
    [0.0, 1.0, 0.0, -1.0],
    [0.0, -1.0, 0.0, 1.0, 0.0, -0.2],
    [0.0, -0.016, 0.0, 1.3, -1.0],
    [0.0, -0.016, 0.0, 0.7, -1.0],
]
diffusion_coefficients = [
    # [epsilon_0, epsilon_1] -> diffu = sqrt(ep0 + ep1 x^2)
    [0.3, 0.1],  # CHECK IF APPROPRIATE
    [0.3, 0.1],  # CHECK IF APPROPRIATE
    [1e-3, 0.1],  # CHECK IF APPROPRIATE
    [1e-5, 0.1],  # CHECK IF APPROPRIATE
]

In [ ]:
def cost_reg(
    xi: np.ndarray,
    KM: tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray],
    lib_drift: np.ndarray,
    lib_diffu: np.ndarray,
    sfp: SteadyFP,
    alpha: float,
    reg_func: Callable[[np.ndarray, np.ndarray], float],
) -> float:
    centers, pdf, drift, diffusion = KM

    drift_val = lib_drift.T @ xi[: len(lib_drift)]
    diffu_val = lib_diffu.T @ xi[len(lib_drift) :]

    drift_fid = np.nansum(drift - drift_val)
    diffusion_fid = np.nansum(diffusion - diffu_val)
    KM_fid = drift_fid + diffusion_fid
    if alpha != 0.0:
        pdf_lib = sfp.solve(drift_val, diffu_val)
        reg_val = reg_func(pdf_lib, pdf)
    else:
        reg_val = 0.0

    return (1 - alpha) * KM_fid + alpha * reg_val


def cost_KL(xi, KM, lib_drift, lib_diffu, sfp, alpha):
    return cost_reg(xi, KM, lib_drift, lib_diffu, sfp, alpha, kl_divergence)


def cost_Jef(xi, KM, lib_drift, lib_diffu, sfp, alpha):
    return cost_reg(xi, KM, lib_drift, lib_diffu, sfp, alpha, jeffreys_divergence)


def cost_Wasserstein(xi, KM, lib_drift, lib_diffu, sfp, alpha):
    return cost_reg(xi, KM, lib_drift, lib_diffu, sfp, alpha, wasserstein_distance)


def opt_func(cost, xi0, KM, lib_drift, lib_diffu, sfp, alpha):
    res = minimize(
        partial(
            cost, KM=KM, lib_drift=lib_drift, lib_diffu=lib_diffu, sfp=sfp, alpha=alpha
        ),
        xi0,
        method="nelder-mead",
        options={"adaptive": True},
    )
    return res.x, res.fun

In [ ]:
method_names = ["KM", "KM KL", "KM Jef", "KM Wasserstein", "KL", "Jef", "Wasserstein"]
alpha_vals = [0.0, 0.5, 0.5, 0.5, 1.0, 1.0, 1.0]
opt_funcs = [
    partial(opt_func, cost_KL),
    partial(opt_func, cost_KL),
    partial(opt_func, cost_Jef),
    partial(opt_func, cost_Wasserstein),
    partial(opt_func, cost_KL),
    partial(opt_func, cost_Jef),
    partial(opt_func, cost_Wasserstein),
]

In [ ]:
NUM_DATASETS = 10
NUM_VALIDATION = 1
NUM_CPUS = os.cpu_count() or 1
dt = 0.001
num_bins = 100
target_metadata = {
    "num_datapoints": 10_000_000,
    "dt": dt,
    "EVEN_ABS": True,
    "coeffs": None,  # fill in later for each equation
    "ep0": None,  # fill in later for each equation
    "ep1": None,  # fill in later for each equation
    "x0": 0.0,
    "num_bins": num_bins,
}

In [ ]:
def timeseries_plots(folder, timeseries, suffix=""):
    time_stack, x_stack = timeseries
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(time_stack, x_stack)
    ax.set_xlabel("Time, $t$")
    ax.set_ylabel("Value, $x$")

    if suffix:
        fig.savefig(folder / f"timeseries_{suffix}.png")
    else:
        fig.savefig(folder / f"timeseries.png")


def KM_plots(folder, KM, suffix=""):
    centers, pdf_stack, drift_stack, diffusion_stack = KM
    fig, (ax1, ax2, ax3) = plt.subplots(3, sharex=True)

    ax1.plot(centers, pdf_stack.T, alpha=0.7)
    ax1.set_ylabel("PDF, $P(x)$")

    ax2.plot(centers, drift_stack.T, alpha=0.7, linestyle="", marker="x")
    ax2.set_ylabel("Drift, $m^{(1)}(x)$")

    ax3.plot(centers, diffusion_stack.T, alpha=0.7, linestyle="", marker="x")
    ax3.set_ylabel("Diffusion, $m^{(2)}(x)$")
    ax3.set_xlabel("$x$")

    fig.tight_layout()

    if suffix:
        fig.savefig(folder / f"kramers_moyal_{suffix}.png")
    else:
        fig.savefig(folder / "kramers_moyal.png")


def KM_plots_one_method(
    folder, KM, found_pdf, found_drift, found_diffusion, method_name: str, suffix=""
):
    centers, pdf_stack, drift_stack, diffusion_stack = KM
    fig, (ax1, ax2, ax3) = plt.subplots(3)
    ax1.set_title(method_name)
    ax1.plot(centers, pdf_stack.T, alpha=0.7, linestyle="", marker="x")
    ax1.plot(centers, found_pdf, linestyle="-", label="Model")
    ax1.set_ylabel("PDF, $P(x)$")

    ax2.plot(centers, drift_stack.T, alpha=0.7, linestyle="", marker="x")
    ax2.plot(centers, found_drift, linestyle="-")
    ax2.set_ylabel("Drift, $m^{(1)}(x)$")

    ax3.plot(centers, diffusion_stack.T, alpha=0.7, linestyle="", marker="x")
    ax3.plot(centers, found_diffusion, linestyle="-")
    ax3.set_ylabel("Diffusion, $m^{(2)}(x)$")

    fig.legend()
    fig.tight_layout()

    if suffix:
        fig.savefig(folder / f"kramers_moyal_{method_name}_{suffix}.png")
    else:
        fig.savefig(folder / f"kramers_moyal_{method_name}.png")


def KM_plots_all_methods(
    folder, KM, found_pdfs, found_drifts, found_diffusions, method_names, suffix=""
):
    # I think this plot will look extremely messy
    centers, pdf_stack, drift_stack, diffusion_stack = KM
    fig, (ax1, ax2, ax3) = plt.subplots(3)
    ax1.plot(centers, pdf_stack.T, alpha=0.7, linestyle="", marker="x")
    ax1.set_ylabel("PDF, $P(x)$")

    ax2.plot(centers, drift_stack.T, alpha=0.7, linestyle="", marker="x")
    ax2.set_ylabel("Drift, $m^{(1)}(x)$")

    ax3.plot(centers, diffusion_stack.T, alpha=0.7, linestyle="", marker="x")
    ax3.set_ylabel("Diffusion, $m^{(2)}(x)$")
    for found_drift, found_diffusion, method_name in zip(
        found_drifts, found_diffusions, method_names
    ):
        ax1.plot(centers, found_pdfs, linestyle="-", label=method_name)
        ax2.plot(centers, found_drift, linestyle="-", label=method_name)
        ax3.plot(centers, found_diffusion, linestyle="-", label=method_name)

    fig.legend()
    fig.tight_layout()

    if suffix:
        fig.savefig(folder / f"kramers_moyal_allmethods_{suffix}.png")
    else:
        fig.savefig(folder / f"kramers_moyal_allmethods.png")


def poly_lib(
    x_sym,
    order: int,
    centers: np.ndarray,
    EVEN_ABS: bool = False,
    ODD_ABS: bool = False,
):
    arr = []
    for i in range(order):
        if EVEN_ABS and i % 2 == 0:
            arr.append(sympy.Abs(x_sym) * x_sym ** (i - 1))
        elif ODD_ABS and i % 2 == 1:
            arr.append(sympy.Abs(x_sym) * x_sym ** (i - 1))
        else:
            arr.append(x_sym**i)
    lib_expr = np.array(arr)
    lib_KM = np.empty((order, len(centers)))
    for k in range(order):
        lamb = sympy.lambdify(x_sym, lib_expr[k])
        lib_KM[k] = lamb(centers)

    return lib_expr, lib_KM


def SSR_loop(opt_func, KM, xi0, lib_drift_KM, lib_diffu_KM, sfp, alpha):
    n_drift = len(lib_drift_KM)
    n_terms = n_drift + len(lib_diffu_KM)
    min_xis = np.zeros((n_terms, n_terms - 1))
    min_Vs = np.full((n_terms - 1), np.inf)
    min_xis[:, 0], min_Vs[0] = opt_func(KM, xi0, lib_drift_KM, lib_diffu_KM, sfp, alpha)
    active = np.array(list(range(n_terms)))

    for k in range(1, n_terms - 1):
        params_list = []
        valid_indices = []
        for j in range(len(active)):
            tmp_active = np.delete(active.copy(), j)

            drift_active = tmp_active[tmp_active < n_drift]
            diffu_active = tmp_active[tmp_active >= n_drift] - n_drift
            if len(drift_active) == 0 or len(diffu_active) == 0:
                continue

            params = [
                KM,
                xi0[tmp_active],
                lib_drift_KM[drift_active],
                lib_diffu_KM[diffu_active],
                sfp,
                alpha,
            ]
            params_list.append(params)
            valid_indices.append(j)

        with mp.Pool(NUM_CPUS) as p:
            results = p.starmap(opt_func, params_list)
        xis = []
        Vs = []
        for Xi, V in results:
            xis.append(Xi)
            Vs.append(V)
        min_cost = np.nanargmin(Vs)
        min_idx = valid_indices[min_cost]
        min_V = Vs[min_cost]
        min_xi = xis[min_cost]
        active = np.delete(active, min_idx)
        min_Vs[k] = min_V
        min_xis[k, active] = min_xi

    return min_xis, min_Vs


def eval_jump(series):
    diff = series[1:] - series[:-1]
    max_jump_idx = np.argmax(diff)
    return max_jump_idx

In [ ]:
for equation_number in range(len(equation_names)):
    name = equation_names[equation_number]
    folder = folders[equation_number]
    (SCRATCH_PATH / folder).mkdir(parents=True, exist_ok=True)
    (FIG_PATH / folder).mkdir(parents=True, exist_ok=True)
    drift_coef = drift_coefficients[equation_number]
    ep0, ep1 = diffusion_coefficients[equation_number]
    target_metadata["ep0"] = ep0
    target_metadata["ep1"] = ep1
    target_metadata["coeffs"] = drift_coef
    timeseries, KM, val_timeseries, val_KM = get_timeseries_and_KM(
        SCRATCH_PATH / folder, target_metadata, NUM_DATASETS, NUM_VALIDATION, NUM_CPUS
    )
    timeseries_plots(FIG_PATH / folder, timeseries, folder)
    KM_plots(FIG_PATH / folder, KM, folder)

    # make libraries for drift and diffusion
    x_sym = sympy.symbols("x")
    num_drift = len(drift_coef) + 2
    lib_drift_expr, lib_drift_KM = poly_lib(x_sym, num_drift, KM[0], True, False)
    num_diffusion = 5
    lib_diffu_expr, lib_diffu_KM = poly_lib(x_sym, num_diffusion, KM[0], False, False)
    xi0 = np.random.normal(0, 1, num_drift + num_diffusion)

    sfp = SteadyFP(num_bins, KM[0][1] - KM[0][0])

    # regress each method against the same dataset with the same library functions,
    # the same initial conditions, and record the resulting final equation
    # do some plots of the final equation against the datasets (and save the data)
    found_pdfs = []
    found_drifts = []
    found_diffus = []
    for method_number in range(len(method_names)):
        method_name = method_names[method_number]
        alpha_val = alpha_vals[method_number]
        opt_func = opt_funcs[method_number]

        xis, Vs = SSR_loop(
            opt_func, KM, xi0, lib_drift_KM, lib_diffu_KM, sfp, alpha_val
        )
        best_xi = xis[eval_jump(Vs)]

        drift_xi = best_xi[:num_drift]
        diffu_xi = best_xi[num_drift:]

        found_drift = lib_drift_KM.T @ drift_xi
        found_diffu = lib_diffu_KM.T @ diffu_xi
        found_pdf = sfp.solve(found_drift, found_diffu)

        found_drift_expr = lib_drift_expr.T @ drift_xi
        found_diffu_expr = lib_diffu_expr.T @ diffu_xi
        print(
            f"dx = ({sympy.N(found_drift_expr, 2)}) dt + {sympy.sqrt(sympy.N(2*found_diffu_expr, 2))}dW"
        )
        print(
            f"$dx = ({sympy.latex(sympy.N(found_drift_expr, 2))}) dt + {sympy.latex(sympy.sqrt(sympy.N(2*found_diffu_expr, 2)))} dW$"
        )

        KM_plots_one_method(
            FIG_PATH / folder,
            KM,
            found_pdf,
            found_drift,
            found_diffu,
            method_name,
            folder,
        )
        found_pdfs.append(found_pdf)
        found_drifts.append(found_drift)
        found_diffus.append(found_diffu)
    KM_plots_all_methods(
        FIG_PATH / folder,
        KM,
        found_pdfs,
        found_drifts,
        found_diffus,
        method_names,
        folder,
    )